# X (Twitter) Veri Seti - Duygu Analizi Projesi
**Begüm ÇETİN — Staj Görevi**

## İçindekiler
1. Kütüphaneler ve Kurulum
2. Keşifsel Veri Analizi (EDA)
3. Spam/Reklam Tespiti ve Temizliği (V2 ve Pilot v5)
4. Ön İşleme (Preprocessing) ve Kinaye Tespiti
5. Modelleme (savasy ve TurkishBERTweet Karşılaştırması)
6. Değerlendirme
7. Sonuç ve Çıktı Dosyası (V2)
8. V2 ve Pilot v5 Verilerini Birleştirme
9. savasy Modelinin Gold-Standard Veriyle Fine-Tune Edilmesi

## Bölüm 1 — Kütüphaneler ve Kurulum

In [1]:
import pandas as pd
import re #metinler içinde arama, bulma ve düzenleme islemleri için düzenli ifadeler (Regex) kullanımı saglar
import string #lower, upper gibi fonksiyonlarin kullanımı icin
from collections import Counter #elemanlarin frekansını saymak icin

try:
    import emoji
except ImportError:
    %pip install emoji
    import emoji

try:
    import nltk  # doğal dil işleme için gerekli olan kütüphane
except ImportError:
    %pip install nltk
    import nltk

try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

from nltk.corpus import stopwords

In [2]:
# dosya Downloads klasorunde, tam yol ile okunuyor
df = pd.read_excel("/Users/begumcetin/Desktop/sentiment analysis ito/veri/X_Tweet_dataset.xlsx")  # ham tweet verisini excel'den oku
print(df.shape)  # satir ve sutun sayisini yazdir
df.head()  # ilk 5 satiri goster

(434, 12)


,id,created_at,date,time,text,lang,hashtag_count,retweet_count,like_count,reply_count,quote_count,author_id
0,2054920537768443982,2026-05-14T13:43:29.000Z,2026-05-14,13:43,"Lizzie,Londra ve İstanbul’daki fiyatları karş...",tr,0.0,4,4,0,0,1276607214367751936
1,2054918753742114997,2026-05-14T13:36:24.000Z,2026-05-14,13:36,İstanbul Bebek'te kapıcı dairesine talep edile...,tr,0.0,0,0,0,0,1895060126626914048
2,2054918432647160049,2026-05-14T13:35:07.000Z,2026-05-14,13:35,Türk Hava Yolları İstanbul Havalimanı İç Hatla...,tr,0.0,0,2,0,0,1702235443
3,2054917617933029553,2026-05-14T13:31:53.000Z,2026-05-14,13:31,İstanbul trafiği artık dayanılmaz bir hal aldı...,tr,2.0,0,0,0,0,1569791343500866048
4,2054913996474425553,2026-05-14T13:17:29.000Z,2026-05-14,13:17,"Değerli Taksi Esnafımız,\n\nTaksi sektörü, gel...",tr,0.0,3,10,2,0,1045348990328425984


## Bölüm 2 — Keşifsel Veri Analizi (EDA)

In [3]:
# genel boyut ve eksik değerler
print(df.shape)
print(df.isnull().sum())

(434, 12)
id                0
created_at        0
date              0
time              0
text              0
lang              0
hashtag_count    50
retweet_count     0
like_count        0
reply_count       0
quote_count       0
author_id         0
dtype: int64


In [4]:
# tekrar eden tweetler
print("Tekrar sayısı:", df["text"].duplicated().sum())

Tekrar sayısı: 2


In [5]:
# ham veride en sık geçen kelimeler
all_words = " ".join(df["text"].astype(str)).split()
print(Counter(all_words).most_common(20))

[('İstanbul', 299), ('ve', 192), ('bir', 131), ('#istanbul', 65), ('için', 62), ('TL', 48), ('da', 43), ('trafik', 41), ('Turizm', 41), ('-', 39), ('ile', 35), ('bin', 33), ('de', 32), ('bu', 32), ('Mayıs', 29), ('Vize', 29), ('en', 28), ('istanbul', 27), ('📍', 26), ('Sayın', 26)]


## Bölüm 3 — Spam/Reklam Tespiti ve Temizliği

In [6]:
#sürekli tekrarlayan spam tweetle karsılastım ve cıkardım
spam_mask = df["text"].str.contains("乂esCort乂|乂esKort乂", regex=True, na=False)
df_clean = df[~spam_mask].copy() #sadece spam OLMAYAN satırları sec
print("önceki satır sayısı: ", df.shape[0])
print("spam çıkarıldıktan sonra: ", df_clean.shape[0])

önceki satır sayısı:  434
spam çıkarıldıktan sonra:  411


In [7]:
# hashtag_count boş olan satırlar incelenirken reklam fark edildi
# incelemede 9 tanesinin gerçek reklam (masaj hizmeti, kredi danışmanlığı,
# konser bileti satışı, at yarışı bülteni), geri kalan 18'i gerçek haber içeriği olduğu
# için dokunulmadı.
extra_spam_keywords = r'masaj hizmeti|fetiş|mistress|Emekliye kredi|At Yarışı Bülteni|Bilet.*[Ss]atılıktır|BİLETLER MEVCUTTUR'
extra_spam_mask = df_clean["text"].str.contains(extra_spam_keywords, regex=True, na=False, case=False)
#na=False => boş hücreye rastlarsan False (spam değil) say.
#case=False => büyük küçük harf duyarsız arama.
#regex=Trıe => düz metin olarak değil regex olarak yorumla.

print("Yeni bulunan spam satır sayısı:", extra_spam_mask.sum()) #sum kaç tane satırın True(spam) olduğunu sayar
df_clean = df_clean[~extra_spam_mask].copy() #spam OLMAYANLARI göster
print("Güncel satır sayısı:", df_clean.shape[0])

Yeni bulunan spam satır sayısı: 9
Güncel satır sayısı: 402


In [8]:
# "AYIPAYIP" reklamı ayrıca bir spam/reklam örneği olarak tespit edildi aynı hesaptan 2 kez atılmış
ayipayip_mask = df_clean["text"].str.contains("AYIPAYIP", regex=False, na=False)
print("AYIPAYIP satır sayısı:", ayipayip_mask.sum())
df_clean = df_clean[~ayipayip_mask].copy()

# geriye kalan gerçek içerik tekrarlarını (aynı hesabın aynı duyuruyu iki kez atması gibi)
# standart yöntemle temizle - her tekrar grubundan ilk gördüğümüzü tutuyoruz
oncesi = df_clean.shape[0]
df_clean = df_clean.drop_duplicates(subset="text", keep="first").copy()
#subset => text sütununa bakark tekrar ara
#keep="first" => bir tekrar grubu bulunca ilkini tut diğerlerini sil
print("Tekrar temizliği öncesi:", oncesi, "sonrası:", df_clean.shape[0])

AYIPAYIP satır sayısı: 2
Tekrar temizliği öncesi: 400 sonrası: 399


In [9]:
onceki_bos_indexler = df[df["hashtag_count"].isnull()].index
hala_var_olanlar = df_clean.index.intersection(onceki_bos_indexler)
#intersection => kesişim(her iksiinde de boş olan satırlar)

for i in hala_var_olanlar:
    print(i, "--", df_clean.loc[i, "text"])
    print("---")

186 -- 50. Yıl kütüphaneleri̇ toplu açılış töreni gerçekleştirildi
https://t.co/3I1Jp3ZujY
@Yusuf__Tekin @gul_davut 
#eğitim #kütüphane #istanbul https://t.co/r87i8RgRu9
---
193 -- Eminönü Esenler 🏃ardali 👦 Etiler #istanbul Maslak değerlendirememe 🅱yastiktakoz https://t.co/xOABwM6wQU
---
199 -- 🌛 senkronik bakırköy başakşehir istanbul #istanbul https://t.co/BdhMPv6t64
---
213 -- #istanbul bebek https://t.co/i7KtlQsBoy
---
221 -- Eminönü lokmanruhu🍉 🦝 Etiler #istanbul Maslak Esenler https://t.co/74SbFsSBIS
---
224 -- Öğrenmek için tıklayın👎👎
DETAY 👉 https://t.co/FSZA0S86cP
#altın #sondakika #sağlık #SONDAKİKA #mayıs #istanbul #pazartesi https://t.co/poXHbefBSD
---
232 -- #istanbul bakırköy derlemek 🈵 istanbul kavuklu başakşehir https://t.co/qwe2gMbA1o
---
261 -- Motorine zam yolda: Yarından itibaren geçerli olacak https://t.co/UKpg2chirL @kokmedyahaber 

#Motorin #Zam #Akaryakıt #SonDakika #Haber #Gündem #KökMedyaHaber #Ekonomi #Benzin #Petrol #Dolar #HızlıHaber #İstanbul #Ankara #İzmir

In [10]:
# manuel incelenip bot/clickbait içerikli olduğu tespit edilen satırlar
# 193, 199, 221, 232 -> anlamsız kelime dizisi + rastgele semt ismi
# 224 -> clickbait
# 297 -> etkileşim isteyen hesap
manuel_supheli_indexler = [193, 199, 221, 224, 232, 297]

oncesi = df_clean.shape[0]
df_clean = df_clean.drop(index=manuel_supheli_indexler).copy()
print("Öncesi:", oncesi, "Sonrası:", df_clean.shape[0])

Öncesi: 399 Sonrası: 393


### Son Spam Kontrolü

In [11]:
print(df_clean["author_id"].value_counts().head(10))

author_id
1935220694566191104    12
1123197386178925056     6
2044467328004263936     5
916819249               5
1997315921946545920     5
2023068080729980928     4
1814782913709232128     4
540837233               3
1585698253433146880     3
1937192006972997888     3
Name: count, dtype: int64


In [12]:
reklam_kalip = r'whatsapp|WhatsApp|iletişim|sipariş|kampanya|indirim|fiyat için|dm at|05\d{9}'
supheli = df_clean["text"].str.contains(reklam_kalip, regex=True, na=False, case=False)
print("Şüpheli satır sayısı:", supheli.sum())
for i, t in df_clean.loc[supheli, "text"].items():
    print(i, "--", t)
    print("---")

Şüpheli satır sayısı: 7
46 -- İBB davası sanığının, İstanbul Cumhuriyet Başsavcılığı’na tahsis ettiği 4 lüks aracın 2’si, resmi görev olmaksızın Ankara’ya gönderilip şahsi kullanım için tahsis edilmiş. Ankara’da trafik cezası kesilince foyaları ortaya çıkmış.
İşte belgesi! 
#CHPiletişim https://t.co/2igvXIzBqe
---
162 -- İstanbul Bakırköy’de bir kurye ile taksici arasında trafikte tartışma çıktı. Daha sonra kurye markete uğrayarak siparişleri almak istedi. Taksici ise arkasından gelerek market içerisinde tartıştı. 

İddiaya göre, bir market personeli ise taksiciye bıçak çekerek tartışmaya dahil https://t.co/MFhDf5DUmL
---
163 -- İstanbul Bakırköy’de bir kurye ile taksici arasında trafikte tartışma çıktı. Daha sonra kurye markete uğrayarak siparişleri almak istedi. Taksici ise arkasından gelerek market içerisinde tartıştı. 

İddiaya göre, bir market personeli ise taksiciye bıçak çekerek tartışmaya dahil https://t.co/1sJyst9ZPx
---
164 -- İstanbul Bakırköy’de bir kurye ile taksici arasın

In [13]:
#aşırı hashtag kullanan tweetleri tespit
print(df_clean.sort_values("hashtag_count", ascending=False)[["text","hashtag_count"]].head(10))

                                                  text  hashtag_count
70   İtirafçının anıları...\nİstanbul\nCHP #İBB İET...            5.0
77   Yeni İş İlanı: Fairmont Quasar Istanbul - Kuru...            5.0
147  İstanbul Şişli’de bir ticari taksi sürücüsü, g...            5.0
268  İstanbul Başakşehir Millet Bahçesi'ndeki araç ...            5.0
118  🗺️ İstanbul'daki trafik yoğunluğu %80'a ulaştı...            5.0
420  İstanbul Sarıyer'de 74 yaşındaki T.K., evinin ...            5.0
127  Elite World, 50 yıllık deneyimini "Bir Dünya V...            5.0
109  Borsa İstanbul günü sert düşüşle kapattı!\nEnf...            5.0
390  ✈️ Fırsat Uçuşu!\n📉 Ortalamadan 2.190 TL daha ...            5.0
374  Rayİst artık size mail gönderiyor 🚇📩\n\nFavori...            5.0


In [14]:
# 1935220694566191104 nolu hesap 12 tweetinin hepsinde vize randevu tweeti paylasmis
vize_bot_mask = df_clean["author_id"] == 1935220694566191104
print("Vize botu satır sayısı:", vize_bot_mask.sum())
df_clean = df_clean[~vize_bot_mask].copy()
print("Güncel satır sayısı:", df_clean.shape[0])

Vize botu satır sayısı: 12
Güncel satır sayısı: 381


In [15]:
# "ucuz uçuş bildirimi" formatında tek bir reklam tweeti
df_clean = df_clean.drop(index=[390], errors="ignore").copy()
#errors="ignore" => hücreyi iki kez calıstırınca hata aldım bu yuzden ekledim
print("Güncel satır sayısı:", df_clean.shape[0])

Güncel satır sayısı: 380


In [16]:
# aynı içeriğin sadece link/boşluk farkıyla birden fazla kez atılıp atılmadığını kontrol ettim
normalized = df_clean["text"].apply(lambda t: re.sub(r"\s+", " ", re.sub(r"http\S+", "", str(t))).strip())
near_dup_mask = normalized.duplicated(keep=False)
#   keep=False -> bir tekrar grubunun TAMAMINI işaretle
print("Near-duplicate olan toplam satır sayısı:", near_dup_mask.sum())

Near-duplicate olan toplam satır sayısı: 24


In [17]:
# her gruptan ilkini tut, geri kalanını çıkar
oncesi = df_clean.shape[0]
df_clean = df_clean[~normalized.duplicated(keep="first")].copy()
print("Near-duplicate temizliği öncesi:", oncesi, "sonrası:", df_clean.shape[0])

Near-duplicate temizliği öncesi: 380 sonrası: 367


In [18]:
df_clean["hashtag_count"] = df_clean["text"].str.count("#")
print(df_clean["hashtag_count"].isnull().sum())

0


## Bölüm 3.1 — Pilot v5 (Yeni) Veri Setinin Temizliği


In [ ]:
df_v5 = pd.read_excel("/Users/begumcetin/Desktop/sentiment analysis ito/veri/pilot_v5_tweets.xlsx")
print("başlangıç:", df_v5.shape[0])

# ADIM 1 - DEDUP 
df_v5 = df_v5.drop_duplicates(subset="id", keep="first").copy()
normalized = df_v5["text"].apply(lambda t: re.sub(r"\s+", " ", re.sub(r"http\S+", "", str(t))).strip())
df_v5 = df_v5[~normalized.duplicated(keep="first")].copy()
print("dedup sonrası:", df_v5.shape[0])

# ADIM 2 - BİLİNEN ALAKASIZ KALIPLAR 
alakasiz_kaliplar = (
    r'icradan sat|Ahbap Soruşturmasında|Haluk Levent|'          # turizm: otel icra + Ahbap
    r'cansız beden|kadın cesedi|PRZMA|ÖLÜM GÜNÜ|vefatının.*yıl|'  # turizm: ceset + PRZMA + vefat/anma
    r'[Tt]rafik [Dd]enetleme.*Sicil|Polis İntiharı|'              # ulaşım: polis intiharı
    r'[Kk]urultay|hizmetçinin getirdigi kahve|'                   # enflasyon: kurultay + CIA/kahve
    r'VESUVIUS|Kone Petro|Taş Beşik|MAYA AI'                     # elle bulunan: Vesuvius, Beşiktaş trivia, Migros AI
)
df_v5 = df_v5[~df_v5["text"].str.contains(alakasiz_kaliplar, regex=True, na=False, case=False)].copy()

# ADIM 3 - "İSTANBUL" HİÇ GEÇMEYEN TWEETLERİ ÇIKAR (Türkçe İ karakteri sorunu için [İIiı] kullanıldı)
istanbul_yok = ~df_v5["text"].str.contains(r'[İIiı]stanbul', case=False, na=False, regex=True)
print("İstanbul geçmeyen:", istanbul_yok.sum())
df_v5 = df_v5[~istanbul_yok].copy()

print("SONUÇ:", df_v5.shape[0])
print(df_v5["topic"].value_counts())

# ADIM 4 - Boğaziçi "nasıl oluştu" tekrarı 
df_v5 = df_v5[df_v5["id"] != 2075946418217382179].copy()

# ADIM 5 - ilk 100 taramasında elle bulunan alakasız/reklam/spam tweetler
elle_bulunan_kaliplar = (
    r'F19\.İstanbul Fatih Satılık|'                # emlak ilanı (telefonlu)
    r'25!senedir Akp yönetmiş|'                     # AKP/CHP maaş-kira siyasi eleştiri
    r'premiere of the movie Chicago|'               # film galası trivia
    r'kampusünenyakışıklısı|'                       # üniversite nostaljisi
    r'Değerli abilerim kardeşlerim iş arıyorum|'    # bireysel iş arama çağrısı
    r'Cuma Mesai çıkışı yollara düşüp|'             # Bodrum/Kuşadası paralı beach yorumu
    r'ŞOK OĞUZHAN DTADLI MARKET|'                   # market şikayeti
    r'FENERBAHÇE YİNE FİKSTÜR ŞİKESİ|'              # Fenerbahçe fikstür 1
    r'en büyük fikstür hilesi yapıldı|'             # Fenerbahçe fikstür 2
    r'SelamlıQue İstanbul markası|'                 # kişisel ürün yorumu
    r'İBB Kremlin kadrolarında|'                    # İBB mühendis yolsuzluk iddiası
    r'maddidestek verip kart borcunu|'              # spam/dolandırıcılık
    r'1071 Malazgirt|'                              # tarih+enflasyon şakası
    r'GERÇEK OLUYOR!#gogabeach'                     # Goga Beach reklamı
)
elle_sayisi = df_v5["text"].str.contains(elle_bulunan_kaliplar, regex=True, na=False, case=False).sum()
print("elle bulunan alakasız/reklam sayısı:", elle_sayisi)
df_v5 = df_v5[~df_v5["text"].str.contains(elle_bulunan_kaliplar, regex=True, na=False, case=False)].copy()

print("SONUÇ:", df_v5.shape[0])
print(df_v5["topic"].value_counts())

# ADIM 6 - spam
df_v5 = df_v5[~df_v5["id"].isin([2075329029259817282, 2074488453861925312])].copy()

# ADIM 7 - ikinci tarama: alakasız/reklam/duplicate tweetler
ikinci_tur_kaliplar = (
    r'üzcuy beyzuz|'                                # anlamsız/tutarsız post
    r'tatilDARBE ŞAİRİ|'                             # siyasi + anlaşılmaz
    r'Mason Greenwood|'                              # futbol transferi
    r'milletin bikinili tatil storylerine|'          # zayıf kişisel içerik
    r'AntalyaBasket\'in menajeriydi|'                # vefat/anma
    r'evli çift müşteriye giderken|'                 # örtük hizmet reklamı
    r'İKMİB kaynıyor|'                                # örgüt yolsuzluğu, turizm değil
    r'PERA PALAS OTELDE kalan İngiliz|'              # tarih trivia + siyasi
    r'Playliste Ikınarak|'                           # müzik eleştirisi
    r'esenyurtta otel de görüşme|'                   # örtük hizmet reklamı
    r'viyadük girişine otobüs saatte|'               # FETÖ komplosu (2 kopya)
    r'BİR BARINAK KAPSINDA AĞLAMASIN|'               # hayvan sahiplendirme
    r'Terk edilen Efe yuvasını arıyor'                # hayvan sahiplendirme
)
ikinci_tur_sayisi = df_v5["text"].str.contains(ikinci_tur_kaliplar, regex=True, na=False, case=False).sum()
print("ikinci tur alakasız sayısı:", ikinci_tur_sayisi)
df_v5 = df_v5[~df_v5["text"].str.contains(ikinci_tur_kaliplar, regex=True, na=False, case=False)].copy()

print("SONUÇ:", df_v5.shape[0])
print(df_v5["topic"].value_counts())


# ADIM 8 - Claude'un tarafında yapılan ikinci tur tarama: duplicate haber kümeleri + yeni alakasız kalıplar
ikinci_tarama_id_listesi = [
    # Torunlar GYO "gunluk kira 44 milyon TL" haberi - 6 kopyadan 5'i
    2074980156301554057, 2075294384673771740, 2074950794038616202, 2074871393687150677, 2074881882068505013,
    # Istanbul Tabip Odasi katilim payi haberi - duplicate
    2074589730033799534,
    # kira %40/45bin TL haberi - duplicate
    2076330590551392476,
    # IETT Mecidiyekoy otobus-yaya kazasi - 6 kopyadan 5'i
    2075253756338335914, 2075255361968226367, 2075297696949125288, 2075337644012097595, 2075256986392727661,
    # AJet Trabzon sigara olayi - 3 kopyadan 2'si
    2076047233934332384, 2076045402361778401,
    # 15 Temmuz ucretsiz ulasim duyurusu - 9 kopyadan 8'i
    2075702705423614021, 2075712450356490647, 2076001516628893865, 2076012575763136565, 2076028753390522613,
    2076046771789209750, 2076052325857128487, 2076556943234081052,
    # FETO/Fenerbahce otobus komplosu - siyasi komplo + duplicate (3 kopya)
    2075684338994418038, 2075687990202872111, 2076008710648799655,
    # Orhan Aydin NATO protestosu metro - siyasi protesto + duplicate
    2074860169456939457, 2074860843255173276, 2074577951362801767, 2074611107851837890,
    # Usкudar Marmaray raylara inen kadin - duplicate
    2076040424410763770,
    # Simitci 120TL olayi - 3 kopyadan 2'si
    2075901384520519900, 2076048379998564852,
    # Bogaz teleferik onerisi - duplicate
    2076040077906768156,
    # Macaristan Basbakani Peter Magyar Istanbul ziyareti - siyasi/diplomatik haber + duplicate kumesi (7 tweet)
    2074933381918511164, 2074927534966288597, 2074926064758108570, 2074424159992508897,
    2074602710532329811, 2074621748377506293, 2076048704444793274,
    # tekil alakasiz bulgular: hayvan kurtarma, kurumsal is birligi, TMSF hastane satisi, iscil sendika/eylem,
    # unlu dedikodusu, tarihi/diplomatik trivia, siyasi parti orgut ziyareti, is ilani, reklam
    2074626644091019555, 2074932083886084262, 2075602445225586972, 2075620544330739817, 2075920195525259310,
    2074510915601420425, 2074585889284501952, 2075293731779957148, 2075632478421745949, 2075597033616945267,
    2074459664415363459, 2074897556643995910, 2076023198685253795, 2075577995092218139, 2075942571260469575,
    2076270319128432670, 2075218061997531165, 2075307703488897209, 2076637829161992239, 2076042147153715581,
    2076579922198307191, 2074909519197438199, 2075262768593162583, 2075292838506459552, 2074925725417718165,
]
print("ikinci tarama ile cikacak:", df_v5["id"].isin(ikinci_tarama_id_listesi).sum())
df_v5 = df_v5[~df_v5["id"].isin(ikinci_tarama_id_listesi)].copy()

# ADIM 9 - hayvan sahiplendirme kalibindaki yazim hatasini duzelt 
hayvan_mask = df_v5["text"].str.contains(r'BARINAK KAP.SINDA AĞLAMASIN', regex=True, na=False, case=False)
print("kacırılan hayvan sahiplendirme:", hayvan_mask.sum())
df_v5 = df_v5[~hayvan_mask].copy()

print("SONUÇ:", df_v5.shape[0])
print(df_v5["topic"].value_counts())

başlangıç: 399
dedup sonrası: 374
İstanbul geçmeyen: 46
SONUÇ: 295
topic
ulasim       103
turizm        98
enflasyon     94
Name: count, dtype: int64
elle bulunan alakasız/reklam sayısı: 11
SONUÇ: 283
topic
ulasim       103
turizm        97
enflasyon     83
Name: count, dtype: int64
ikinci tur alakasız sayısı: 12
SONUÇ: 270
topic
ulasim       100
turizm        88
enflasyon     82
Name: count, dtype: int64
ikinci tarama ile cikacak: 65
kacırılan hayvan sahiplendirme: 1
SONUÇ: 204
topic
ulasim       69
enflasyon    68
turizm       67
Name: count, dtype: int64


In [20]:
# temizlenmiş pilot v5 verisini kontrol etmen için Excel'e kaydet
with pd.ExcelWriter("/Users/begumcetin/Desktop/sentiment analysis ito/ciktilar/pilot_v5_tweets_temiz.xlsx", engine="openpyxl") as writer:
    df_v5.to_excel(writer, index=False, sheet_name="Sonuclar")  # temizlenmis pilot v5'i excel'e yaz
    ws = writer.sheets["Sonuclar"]
    ws.auto_filter.ref = ws.dimensions   # sutunlara filtre oku ekler
    ws.freeze_panes = "A2"                # baslik satirini sabitler

print("Kaydedildi: pilot_v5_tweets_temiz.xlsx")

Kaydedildi: pilot_v5_tweets_temiz.xlsx


## Bölüm 4 — Ön İşleme (Preprocessing)

In [21]:
import re  # regex (duzenli ifade) islemleri icin kutuphane
import emoji  # emoji tespit/temizleme kutuphanesi

def clean_url(text):
  #re.sub(pattern, repl, string)
  #pattern: aranacak desen, repl: değiştirilecek, string: üzerinde işlem yapılacak metin
  #httpyi arar, \S+ = "bosluga kadar devam eden her sey"
  return re.sub(r'http\S+', '', text)  # http ile baslayan url'leri bulup siler

ornek = df["text"][0]  # df'nin ilk satirindaki tweet metnini ornek olarak sec
print("öncesi: ", ornek)  # temizlik oncesi orijinal metni yazdir
print("sonrası: ", clean_url(ornek))  # clean_url uygulanmis halini yazdir

öncesi:  Lizzie,Londra ve İstanbul’daki  fiyatları karşılaştırmış 

İstanbul neredeyse Londra’nın 2 katı daha pahalı https://t.co/5IdhnqVibK
sonrası:  Lizzie,Londra ve İstanbul’daki  fiyatları karşılaştırmış 

İstanbul neredeyse Londra’nın 2 katı daha pahalı 


In [22]:
def clean_mention(text):
  # @ karakterini arar,
  #\w ise \S'den farklı olarak harf, rakam veya alt cizgileri kapsar.
  #\S noktalma isaretlerini de kapsar
  return re.sub(r'@\w+','', text)  # @kullaniciadi seklindeki mention'lari bulup siler

ornek = df_clean["text"][4]  # df_clean'den 4. indeksteki tweeti ornek olarak sec
print("öncesi: ", ornek)  # mention temizligi oncesi metni yazdir
print("sonrası: ", clean_mention(ornek))  # mention'lari silinmis halini yazdir

öncesi:  Değerli Taksi Esnafımız,

Taksi sektörü, geleceği güçlü, yatırım değeri yüksek bir sektördür.

Taksi plakalarınızın geleceğine güvenin, plakalarınızı satmayın.

İSMET DALCI
İSTANBUL TAKSİCİLER ESNAF ODASI BAŞKANI 

@MehmetYiginer 
@tsofresmi 
@istesob https://t.co/jtPxD5lwLp
sonrası:  Değerli Taksi Esnafımız,

Taksi sektörü, geleceği güçlü, yatırım değeri yüksek bir sektördür.

Taksi plakalarınızın geleceğine güvenin, plakalarınızı satmayın.

İSMET DALCI
İSTANBUL TAKSİCİLER ESNAF ODASI BAŞKANI 

 
 
 https://t.co/jtPxD5lwLp


In [23]:
def clean_hashtag_symbol(text):
  #hashtagler de topic belirleyebilecegi icin silinmedi, sadece hashtag isareti silindi
  return text.replace('#','')  # sadece # karakterini siler, hashtag kelimesi metinde kalir

ornek = df_clean["text"][3]  # 3. indeksteki tweeti ornek olarak sec
print("öncesi: ", ornek)  # # isareti silinmeden once metni yazdir
print("sonrası: ", clean_hashtag_symbol(ornek))  # # isareti silinmis halini yazdir

öncesi:  İstanbul trafiği artık dayanılmaz bir hal aldı.

#trafik #istanbul https://t.co/HHMoGBZaQb
sonrası:  İstanbul trafiği artık dayanılmaz bir hal aldı.

trafik istanbul https://t.co/HHMoGBZaQb


In [24]:
def clean_emoji(text):
  return emoji.replace_emoji(text, replace=' ')  # metindeki tum emojileri bosluk ile degistirir

ornek = df_clean["text"][7]  # 7. indeksteki tweeti ornek olarak sec
print("öncesi: ", ornek)  # emoji temizligi oncesi metni yazdir
print("sonrası: ", clean_emoji(ornek))  # emojileri temizlenmis halini yazdir

öncesi:  🐂 Kaçan Dana E-5'i Karıştırdı

Avcılar'da kurban pazarından kaçan dana, E-5'e girdi. Trafik bir süreliğine aksadı.

#Istanbul #Avcilar
sonrası:    Kaçan Dana E-5'i Karıştırdı

Avcılar'da kurban pazarından kaçan dana, E-5'e girdi. Trafik bir süreliğine aksadı.

#Istanbul #Avcilar


In [25]:
def turkish_lower(text):
  text = text.replace('İ', 'i').replace('I','ı')  # Turkce buyuk I/İ karakterlerini Python'un yanlis kucultmesini onlemek icin once manuel degistir
  return text.lower()  # kalan tum harfleri kucuk harfe cevirir

test = "İstanbul IŞIK Trafiği"  # Turkce I/İ sorununu gostermek icin test metni

print("Python .lower():  ", test.lower())  # standart .lower()'in hatali sonucunu goster
print("turkish_lower: ", turkish_lower(test))  # duzeltilmis fonksiyonun dogru sonucunu goster

Python .lower():   i̇stanbul işik trafiği
turkish_lower:  istanbul ışık trafiği


In [26]:
#noktalama işaretlerini temizle
import string  # string.punctuation icin (tum noktalama isaretlerini iceren hazir liste)

def clean_punctuation(text):
  return text.translate(str.maketrans('','', string.punctuation))  # metindeki tum noktalama isaretlerini siler

In [27]:
def preprocess(text):
  text = clean_url(text)  # url'leri sil
  text = clean_mention(text)  # mention'lari sil
  text = clean_hashtag_symbol(text)  # # isaretini sil, kelimeyi birak
  text = clean_emoji(text)  # emojileri sil
  text = turkish_lower(text)  # Turkce kurallarina gore kucuk harfe cevir
  text = clean_punctuation(text)  # noktalama isaretlerini sil
  return text  # tum adimlardan gecmis temiz metni dondur

ornek = df_clean["text"].iloc[4]  # 4. satirdaki tweeti ornek olarak sec
print("öncesi: ", ornek)  # tum on isleme oncesi metni yazdir
print()
print("sonrası: ", preprocess(ornek))  # tum on isleme adimlarindan gecmis halini yazdir

öncesi:  Değerli Taksi Esnafımız,

Taksi sektörü, geleceği güçlü, yatırım değeri yüksek bir sektördür.

Taksi plakalarınızın geleceğine güvenin, plakalarınızı satmayın.

İSMET DALCI
İSTANBUL TAKSİCİLER ESNAF ODASI BAŞKANI 

@MehmetYiginer 
@tsofresmi 
@istesob https://t.co/jtPxD5lwLp

sonrası:  değerli taksi esnafımız

taksi sektörü geleceği güçlü yatırım değeri yüksek bir sektördür

taksi plakalarınızın geleceğine güvenin plakalarınızı satmayın

ismet dalcı
istanbul taksiciler esnaf odası başkanı 

 
 
 


In [28]:
def tokenize(text):
  return text.split()  # metni bosluklardan bolerek kelime listesine (token) cevirir

def remove_stopwords(tokens):
  sonuc = []  # stopword olmayan kelimelerin toplanacagi bos liste
  for word in tokens: #tokens listesindeki tüm kelimeleri gez
    if word not in turkce_stopwords: #stopword listesinde yoksa yeni listeye ekle
      sonuc.append(word)  # stopword degilse listeye ekle
  return sonuc  # stopword'lerden ayiklanmis token listesini dondur

In [29]:
#stopwordsüz halinde bir kelimesinin cıkarılmadıgını gordum bu yuzden stopword listesini kontrol ettim
print(sorted(stopwords.words('turkish')))

# istanbul kelimesinin satırların %92.7'sinde geçtiği sentiment için ayırt edici olmadığı belirlendi
# olan/var/olarak ise işlevsel (gramer) kelimeler oldukları için eklendi
ekstra_stopwordler = {"bir", "rt", "https", "t", "co", "var", "olarak", "istanbul", "olan"}
turkce_stopwords = set(stopwords.words('turkish')) | ekstra_stopwordler
print(len(turkce_stopwords))

['acaba', 'ama', 'aslında', 'az', 'bazı', 'belki', 'biri', 'birkaç', 'birşey', 'biz', 'bu', 'da', 'daha', 'de', 'defa', 'diye', 'en', 'eğer', 'gibi', 'hem', 'hep', 'hepsi', 'her', 'hiç', 'ile', 'ise', 'için', 'kez', 'ki', 'kim', 'mu', 'mü', 'mı', 'nasıl', 'ne', 'neden', 'nerde', 'nerede', 'nereye', 'niye', 'niçin', 'o', 'sanki', 'siz', 'tüm', 've', 'veya', 'ya', 'yani', 'çok', 'çünkü', 'şey', 'şu']
62


In [30]:
ornek = df_clean["text"].iloc[4]
temiz = preprocess(ornek)
tokenlar = tokenize(temiz)
tokenlar_stopwordsuz = remove_stopwords(tokenlar)

print("orijinal: ", ornek)
print()
print("preprocess sonrası temiz metin: ", temiz)
print()
print("tokenlar: ", tokenlar)
print()
print("stopwordsüz tokenlar: ", tokenlar_stopwordsuz)

orijinal:  Değerli Taksi Esnafımız,

Taksi sektörü, geleceği güçlü, yatırım değeri yüksek bir sektördür.

Taksi plakalarınızın geleceğine güvenin, plakalarınızı satmayın.

İSMET DALCI
İSTANBUL TAKSİCİLER ESNAF ODASI BAŞKANI 

@MehmetYiginer 
@tsofresmi 
@istesob https://t.co/jtPxD5lwLp

preprocess sonrası temiz metin:  değerli taksi esnafımız

taksi sektörü geleceği güçlü yatırım değeri yüksek bir sektördür

taksi plakalarınızın geleceğine güvenin plakalarınızı satmayın

ismet dalcı
istanbul taksiciler esnaf odası başkanı 

 
 
 

tokenlar:  ['değerli', 'taksi', 'esnafımız', 'taksi', 'sektörü', 'geleceği', 'güçlü', 'yatırım', 'değeri', 'yüksek', 'bir', 'sektördür', 'taksi', 'plakalarınızın', 'geleceğine', 'güvenin', 'plakalarınızı', 'satmayın', 'ismet', 'dalcı', 'istanbul', 'taksiciler', 'esnaf', 'odası', 'başkanı']

stopwordsüz tokenlar:  ['değerli', 'taksi', 'esnafımız', 'taksi', 'sektörü', 'geleceği', 'güçlü', 'yatırım', 'değeri', 'yüksek', 'sektördür', 'taksi', 'plakalarınızın', 'g

In [31]:
def remove_pure_numbers(tokens):
    # sadece TAMAMEN sayidan olusan tokenlari cikar (2026, 2, 35 gibi)
    # 'tl', 'bin' gibi harften olusan kelimelere dokunmuyo
    return [t for t in tokens if not t.isdigit()]  # sadece rakamdan olusmayan tokenlari tutar, sayisal olanlari eler

test_tokens = ["zam", "2026", "yılında", "35", "bin", "tl", "oldu"]  # fonksiyonu denemek icin ornek token listesi
print(remove_pure_numbers(test_tokens))  # sayisal tokenlar cikarilmis halini yazdir

['zam', 'yılında', 'bin', 'tl', 'oldu']


In [32]:
def full_pipeline(text):
  temiz = preprocess(text)  # url/mention/hashtag/emoji/kucuk harf/noktalama temizligini uygula
  tokenlar = tokenize(temiz)  # temiz metni kelimelere bol
  tokenlar = remove_pure_numbers(tokenlar)  # sadece sayidan olusan tokenlari cikar
  tokenlar_stopwordsuz = remove_stopwords(tokenlar)  # stopword'leri cikar
  return tokenlar_stopwordsuz  # tum pipeline'dan gecmis token listesini dondur

df_clean["tokens"]=df_clean["text"].apply(full_pipeline)  # her tweete pipeline'i uygulayip token listesi sutunu olustur
df_clean["processed_text"]=df_clean["tokens"].apply(lambda t: " ".join(t))  # token listesini tekrar tek bir metin stringine birlestir

df_clean[["text","processed_text"]].head(10)  # orijinal ve islenmis metni yan yana goster

,text,processed_text
0,"Lizzie,Londra ve İstanbul’daki fiyatları karş...",lizzielondra istanbul’daki fiyatları karşılaşt...
1,İstanbul Bebek'te kapıcı dairesine talep edile...,bebekte kapıcı dairesine talep edilen yüksek k...
2,Türk Hava Yolları İstanbul Havalimanı İç Hatla...,türk hava yolları havalimanı iç hatlar özel yo...
3,İstanbul trafiği artık dayanılmaz bir hal aldı...,trafiği artık dayanılmaz hal aldı trafik
4,"Değerli Taksi Esnafımız,\n\nTaksi sektörü, gel...",değerli taksi esnafımız taksi sektörü geleceği...
5,Evet rakam az Ama birçok emeklinin de durumlar...,evet rakam birçok emeklinin durumları iyi geçe...
6,Türkiye Seyahat Acentaları Birliği (TÜRSAB) ve...,türkiye seyahat acentaları birliği türsab rusy...
7,🐂 Kaçan Dana E-5'i Karıştırdı\n\nAvcılar'da ku...,kaçan dana e5i karıştırdı avcılarda kurban paz...
8,İstanbul’da Kurban Bayramı öncesi otobüs firma...,istanbul’da kurban bayramı öncesi otobüs firma...
10,İstanbul Bahçelievler’de kredi kartı komisyonu...,bahçelievler’de kredi kartı komisyonu nedeniyl...


In [33]:
# istanbul kelimesinin oranını doğrulama (stopword listesine ekleme kararı)
istanbul_gecen_satir = df["text"].str.contains(r'\bistanbul\b', case=False, regex=True, na=False)
print("İstanbul geçen satır sayısı:", istanbul_gecen_satir.sum())
print("Toplam satır sayısı:", len(df_clean))
print("Oran:", istanbul_gecen_satir.sum() / len(df_clean))

İstanbul geçen satır sayısı: 432
Toplam satır sayısı: 367
Oran: 1.1771117166212535


In [34]:
from collections import Counter
tum_kelimeler = [kelime for token_listesi in df_clean["tokens"] for kelime in token_listesi]
print(Counter(tum_kelimeler).most_common(30))

[('trafik', 63), ('tl', 58), ('zam', 39), ('turizm', 37), ('kira', 33), ('bin', 32), ('enflasyon', 27), ('metro', 26), ('asgari', 26), ('mayıs', 25), ('sayın', 24), ('taksi', 23), ('ziyaret', 23), ('emekli', 22), ('pahalı', 21), ('il', 21), ('yeni', 21), ('otobüs', 20), ('ankara', 20), ('ulaşım', 17), ('ilk', 17), ('sondakika', 17), ('yıl', 16), ('kültür', 16), ('yok', 16), ('başkanımız', 16), ('seyahat', 15), ('maaş', 15), ('otel', 15), ('iki', 15)]


## 4.1 — Kinaye/İroni Şüphesi Tespiti

In [35]:
def sarcasm_supheli(text):
    metin = text.lower()  # karsilastirma icin metni kucuk harfe cevir

    pozitif_kelimeler = ["harika", "süper", "muhteşem", "ne güzel", "tebrikler", "bravo", "aferin", "gurur duyun"]  # kinayede sik kullanilan asiri pozitif kelimeler
    negatif_konu_kelimeleri = ["zam", "kriz", "trafik", "enflasyon", "pahalı", "iptal", "gecikme"]  # aslinda olumsuz olan konu basliklari
    kinaye_emojileri = ["👏", "🙄", "😐", "😑"]  # kinaye/alay belirtebilecek emojiler

    pozitif_var = any(k in metin for k in pozitif_kelimeler)  # metinde pozitif kelimelerden biri var mi kontrol et
    negatif_konu_var = any(k in metin for k in negatif_konu_kelimeleri)  # metinde negatif konu kelimelerinden biri var mi kontrol et
    emoji_var = any(e in text for e in kinaye_emojileri)  # metinde kinaye emojilerinden biri var mi kontrol et

    # aşırı noktalama tek başına yeterli değil, çıkarıldı - yanlış alarm üretiyordu
    return bool((pozitif_var and negatif_konu_var) or (emoji_var and negatif_konu_var))  # pozitif kelime+negatif konu VEYA kinaye emoji+negatif konu varsa kinaye supheli say

df_clean["sarcasm_suspicion"] = df_clean["text"].apply(sarcasm_supheli)  # her tweet icin kinaye supheli olup olmadigini hesapla
supheli_kinaye = df_clean[df_clean["sarcasm_suspicion"]]  # sadece kinaye supheli olan satirlari filtrele
print("Sarcasm şüphesi olan satır sayısı:", len(supheli_kinaye))  # supheli satir sayisini yazdir
for i, t in supheli_kinaye["text"].items():  # supheli her satiri gez
    print(i, "--", t)  # satirin index'ini ve metnini yazdir
    print("---")

Sarcasm şüphesi olan satır sayısı: 2
158 -- Burası neresi tam bilmiyorum ama boşver onlar da bilmesin.. gelmeyin Ankara’ya gözünüzü seviyim gelmeyin.. ıyy çok sıkıcı 🥱 gelmeyin de leş gibi trafik daha da artmasın.. bak İstanbul ne güzel cıvıl cıvıl gidin oraya https://t.co/qTSwhwpddh
---
378 -- İstanbul göbeği Üsküdar’da ıkına sıkıla zorlanarak çeken 5 g internet ağınız ile gurur duyun @TTDestek dünya üzerinde en pahalı internet bizde en karşılığı olmayan hizmet sizde 👏
---


In [36]:
def preprocess_for_bert(text):
  #bu model buyuk kucuk harf ayrımını koruyarak egitildigi icin
  #turkish_lower, clean_punctuation, stopwrod cıkarma kısımlarını buna eklemicez
  text = clean_url(text)  # url'leri sil
  text = clean_mention(text)  # mention'lari sil
  text = clean_hashtag_symbol(text)  # # isaretini sil
  text = clean_emoji(text)  # emojileri sil
  return text  # buyuk/kucuk harf ve noktalama korunmus halde metni dondur

df_clean["bert_input_text"] = df_clean["text"].apply(preprocess_for_bert)  # BERT modeline verilecek hafif temizlenmis metin sutununu olustur
df_clean[["text","bert_input_text"]].head(5)  # orijinal ve BERT-icin-temiz metni yan yana goster

,text,bert_input_text
0,"Lizzie,Londra ve İstanbul’daki fiyatları karş...","Lizzie,Londra ve İstanbul’daki fiyatları karş..."
1,İstanbul Bebek'te kapıcı dairesine talep edile...,İstanbul Bebek'te kapıcı dairesine talep edile...
2,Türk Hava Yolları İstanbul Havalimanı İç Hatla...,Türk Hava Yolları İstanbul Havalimanı İç Hatla...
3,İstanbul trafiği artık dayanılmaz bir hal aldı...,İstanbul trafiği artık dayanılmaz bir hal aldı...
4,"Değerli Taksi Esnafımız,\n\nTaksi sektörü, gel...","Değerli Taksi Esnafımız,\n\nTaksi sektörü, gel..."


## Bölüm 5 — Modelleme
*Sıradaki adım: model ile tahmin üretimi.*

In [37]:
try:
  import transformers #huggingfacein model/tokenizer araclarını iceren ana kutuphane
except ImportError:
  %pip install transformers
  import transformers
 
try:
  import torch
except ImportError:
  %pip install torch
  import torch

from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline

model = AutoModelForSequenceClassification.from_pretrained("savasy/bert-base-turkish-sentiment-cased")
tokenizer = AutoTokenizer.from_pretrained("savasy/bert-base-turkish-sentiment-cased")
sa = pipeline("sentiment-analysis", tokenizer=tokenizer, model=model)

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 10497.25it/s]


In [39]:
def sinifa_cevir(label,score,esik=0.60):
  #notr sınıfı olusturduk
  pozitif_mi=label.lower() == "positive"  # modelin ham etiketi "positive" mi kontrol et
  if score < esik:
      return "nötr"  # guven skoru esigin altindaysa notr say
  elif pozitif_mi:
      return "pozitif"  # esigin ustunde ve pozitifse pozitif dondur
  else:
      return "negatif"  # esigin ustunde ve pozitif degilse negatif dondur

def bert_tahmin_et(text):
  sonuc=sa(text, truncation=True, max_length=128)[0]  # modeli calistir, ilk (tek) sonucu al
  return sinifa_cevir(sonuc["label"],sonuc["score"]),sonuc["score"],sonuc["label"]  # 3 sinifli etiket, guven skoru ve ham etiketi birlikte dondur

sonuclar = df_clean["bert_input_text"].apply(bert_tahmin_et)  # her tweet icin modeli calistir
df_clean["sentiment_label"] = sonuclar.apply(lambda x: x[0])  # 3 sinifli (negatif/notr/pozitif) etiketi sutuna yaz
df_clean["bert_score"] = sonuclar.apply(lambda x: x[1])  # modelin guven skorunu sutuna yaz
df_clean["bert_raw_label"] = sonuclar.apply(lambda x: x[2])  # modelin ham (2 sinifli) etiketini sutuna yaz


print(df_clean["sentiment_label"].value_counts())  # sinif dagilimini yazdir

sentiment_label
negatif    232
pozitif    110
nötr        25
Name: count, dtype: int64


In [40]:
#0.60-0.65 sınırındaki tweetlere manuel bakmak istedim
sinirdaki = df_clean[(df_clean["bert_score"] >= 0.60) & (df_clean["bert_score"] < 0.65)]
for i, row in sinirdaki.iterrows():
    print(row["bert_input_text"][:100], "--", row["bert_raw_label"], row["bert_score"])
    print("---")

Lizzie,Londra ve İstanbul’daki  fiyatları karşılaştırmış 

İstanbul neredeyse Londra’nın 2 katı daha -- negative 0.6063886880874634
---
Türk Hava Yolları İstanbul Havalimanı İç Hatlar Özel Yolcu Salonunu Yeniledi
 
Misafirlerine sunduğu -- positive 0.647861897945404
---
İstanbul’da Kurban Bayramı öncesi otobüs firmalarına yönelik “bilet fiyatı” denetimleri sıklaştırıld -- negative 0.6325721740722656
---
  İstanbul-Singapur 
  Gidiş-Dönüş
  Haziran
  21000-23000 TL
  1 Kabin Bagajı 7 kg +30 kg Uçak Altı -- positive 0.6400262713432312
---
İstanbul Sanayi Odası enflasyon tahmininde revizyon bekliyor, Yönetim Kurulu Başkanı Erdal Bahçıvan: -- negative 0.6295856237411499
---
Fotoğraflarım gerçek, yapay zeka değil İstanbul boğazını ve dolmabahçe sarayını gören, adı "S" ile b -- positive 0.6301349401473999
---
İlgilisine...
13-15 Mayıs tarihleri arasında İstanbul Ticaret Üniversitesi Sütlüce Yerleşkesinde "Ba -- positive 0.6328440308570862
---
 İstanbul

Cumhurbaşkanımız Sayın  ve Sayın  Hanı

In [41]:
#yukarıdaki metinlerin nötr olduğunu düşünüyorum bu yüzden eşiği 0.65e çektim
esik = 0.65
df_clean["sentiment_label"] = df_clean.apply(
    lambda row: sinifa_cevir(row["bert_raw_label"], row["bert_score"], esik=esik), axis=1
)
print(df_clean["sentiment_label"].value_counts())

sentiment_label
negatif    224
pozitif    100
nötr        43
Name: count, dtype: int64


In [42]:
sarcasm_tweetler = df_clean[df_clean["sarcasm_suspicion"]]
print(sarcasm_tweetler[["text", "sentiment_label", "bert_score", "bert_raw_label"]].to_string())

                                                                                                                                                                                                                                                 text sentiment_label  bert_score bert_raw_label
158  Burası neresi tam bilmiyorum ama boşver onlar da bilmesin.. gelmeyin Ankara’ya gözünüzü seviyim gelmeyin.. ıyy çok sıkıcı 🥱 gelmeyin de leş gibi trafik daha da artmasın.. bak İstanbul ne güzel cıvıl cıvıl gidin oraya https://t.co/qTSwhwpddh         negatif    0.999265       negative
378                                                                İstanbul göbeği Üsküdar’da ıkına sıkıla zorlanarak çeken 5 g internet ağınız ile gurur duyun @TTDestek dünya üzerinde en pahalı internet bizde en karşılığı olmayan hizmet sizde 👏         negatif    0.957453       negative


## Bölüm 5.1 — Pilot v5 Verisine Sentiment Skorlama

In [43]:
# V2'de kullanılan ayni on isleme + model + esik (0.65) ile tutarlilik icin
df_v5["bert_input_text"] = df_v5["text"].apply(preprocess_for_bert)  # pilot v5 icin de BERT-icin-temiz metni olustur
df_v5["processed_text"] = df_v5["text"].apply(full_pipeline).apply(lambda t: " ".join(t))  # pilot v5 icin de tam on isleme pipeline'ini uygula
df_v5["sarcasm_suspicion"] = df_v5["text"].apply(sarcasm_supheli)  # pilot v5 icin de kinaye supheli sutununu olustur

def bert_tahmin_et_v5(text):
    sonuc = sa(text, truncation=True, max_length=128)[0]  # modeli calistir
    return sinifa_cevir(sonuc["label"], sonuc["score"], esik=0.65), sonuc["score"], sonuc["label"]  # V2'nin kalibre edilmis esigiyle (0.65) etiketle

sonuclar_v5 = df_v5["bert_input_text"].apply(bert_tahmin_et_v5)  # her pilot v5 tweeti icin modeli calistir
df_v5["sentiment_label"] = sonuclar_v5.apply(lambda x: x[0])  # 3 sinifli etiketi sutuna yaz
df_v5["bert_score"] = sonuclar_v5.apply(lambda x: x[1])  # guven skorunu sutuna yaz
df_v5["bert_raw_label"] = sonuclar_v5.apply(lambda x: x[2])  # ham etiketi sutuna yaz

print(df_v5["sentiment_label"].value_counts())  # sinif dagilimini yazdir

sentiment_label
negatif    117
pozitif     64
nötr        23
Name: count, dtype: int64


## Bölüm 5.3 — TurkishBERTweet ile Karşılaştırmalı Sentiment Skorlama

In [ ]:
import sys
sys.path.append("/Users/begumcetin/Desktop/sentiment analysis ito/model")  # Preprocessor klasorunu import edebilmek icin yolu ekle

import torch
from peft import PeftModel, PeftConfig
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from Preprocessor import preprocess as tbt_preprocess  # TurkishBERTweet'in kendi ozel on isleme fonksiyonu

peft_model_adi = "VRLLab/TurkishBERTweet-Lora-SA"  # kullanilacak LoRA adaptorunun HuggingFace adi
peft_config = PeftConfig.from_pretrained(peft_model_adi)  # adaptorun hangi taban modele ait oldugu bilgisini al
tbt_tokenizer = AutoTokenizer.from_pretrained(peft_config.base_model_name_or_path, padding_side="right")  # taban modelin tokenizer'ini yukle
if getattr(tbt_tokenizer, "pad_token_id") is None:
    tbt_tokenizer.pad_token_id = tbt_tokenizer.eos_token_id  # pad token tanimli degilse eos token ile doldur

id2label_tbt = {0: "negatif", 2: "pozitif", 1: "nötr"}  # modelin sayisal ciktisini Turkce etikete cevirmek icin sozluk
tbt_model = AutoModelForSequenceClassification.from_pretrained(
    peft_config.base_model_name_or_path, return_dict=True, num_labels=len(id2label_tbt), id2label=id2label_tbt
)  # taban modeli 3 sinifli classifier basiyla yukle
tbt_model = PeftModel.from_pretrained(tbt_model, peft_model_adi)  # egitilmis LoRA adaptorunu taban modelin uzerine bindir
tbt_model.eval()  # modeli degerlendirme moduna al

def tbt_tahmin_et(text):
    p = tbt_preprocess(text)  # TurkishBERTweet'in kendi on isleme fonksiyonuyla metni hazirla
    ids = tbt_tokenizer(p, return_tensors="pt", truncation=True, max_length=128)  # metni tokenize et
    with torch.no_grad():
        logits = tbt_model(**ids).logits  # modeli calistir
    probs = torch.softmax(logits, dim=-1)[0]  # logit'leri olasiliga cevir
    label_id = probs.argmax().item()  # en yuksek olasilikli sinifin indeksini al
    return id2label_tbt[label_id], probs[label_id].item(), probs.tolist()  # etiket, guven skoru ve tum olasiliklari dondur

def kalibre_et(df):
    sonuclar = df["text"].apply(tbt_tahmin_et)  # her tweet icin TurkishBERTweet tahminini al
    df["sentiment_label_tbt"] = sonuclar.apply(lambda x: x[0])  # tahmin edilen etiketi sutuna yaz
    df["bert_score_tbt"] = sonuclar.apply(lambda x: x[1])  # guven skorunu sutuna yaz
    df["probs_tbt"] = sonuclar.apply(lambda x: x[2])  # tum 3 sinifin olasiliklarini sutuna yaz

    isimler = ["negatif", "nötr", "pozitif"]
    def ikinci_sinif_ve_margin(probs):
        sirali = sorted(range(3), key=lambda i: -probs[i])  # olasiliklari buyukten kucuge sirala
        return isimler[sirali[1]], probs[sirali[0]] - probs[sirali[1]]  # ikinci en olasi sinifi ve aradaki farki dondur

    ikinci_bilgi = df["probs_tbt"].apply(ikinci_sinif_ve_margin)  # her satir icin ikinci tercih ve farki hesapla
    df["ikinci_sinif_tbt"] = ikinci_bilgi.apply(lambda x: x[0])  # ikinci en olasi sinifi sutuna yaz
    df["margin_tbt"] = ikinci_bilgi.apply(lambda x: x[1])  # birinci ve ikinci tercih arasindaki farki sutuna yaz

    # kalibrasyon: notr dedigi ama ikinci tercihi negatif olan VE aralarindaki fark
    # dar olan (<0.15) tweetler gercekte negatif cikiyor (ambulans/siddet ornekleriyle dogrulandi)
    duzeltme_mask = (
        (df["sentiment_label_tbt"] == "nötr") &  # model notr demis
        (df["ikinci_sinif_tbt"] == "negatif") &  # ama ikinci tercihi negatif
        (df["margin_tbt"] < 0.15)  # ve arada az fark var (kararsiz kalmis)
    )
    df["sentiment_label_tbt_kalibre"] = df["sentiment_label_tbt"]  # once kalibre sutununu ham tahminle baslat
    df.loc[duzeltme_mask, "sentiment_label_tbt_kalibre"] = "negatif"  # kararsiz kalinan satirlari negatife duzelt
    print("duzeltilen tweet sayisi:", duzeltme_mask.sum())  # kac satirin duzeltildigini yazdir
    return df  # kalibre edilmis dataframe'i dondur

df_clean = kalibre_et(df_clean)  # V2 verisine kalibrasyonu uygula
df_v5 = kalibre_et(df_v5)  # pilot v5 verisine kalibrasyonu uygula

print("df_clean - savasy vs TurkishBERTweet (kalibre):")
print(pd.crosstab(df_clean["sentiment_label"], df_clean["sentiment_label_tbt_kalibre"]))  # iki modelin etiketlerini karsilastiran tablo
print()
print("df_v5 - savasy vs TurkishBERTweet (kalibre):")
print(pd.crosstab(df_v5["sentiment_label"], df_v5["sentiment_label_tbt_kalibre"]))

# TBT skoru dusuk olan (0.55 alti) tweetleri, yon ne olursa olsun,
# "manuel kontrol gerekir" olarak isaretle - ozellikle skandal/ciddi haber
# iceriklerinde model kararsiz kalinca yanlis notr diyebiliyor
for df in [df_clean, df_v5]:
    df["tbt_dikkat"] = df["bert_score_tbt"] < 0.55  # guven skoru 0.55'in altindaysa dikkat bayragi koy

print("df_clean - manuel kontrol gereken tweet sayisi:", df_clean["tbt_dikkat"].sum())  # kac satirin dikkat gerektirdigini yazdir
print("df_v5 - manuel kontrol gereken tweet sayisi:", df_v5["tbt_dikkat"].sum())

print()
print("df_clean dikkat gerektirenlerden ornekler:")
print(df_clean[df_clean["tbt_dikkat"]][["text","sentiment_label","sentiment_label_tbt_kalibre","bert_score_tbt"]].head(10).to_string())  # dikkat gerektiren ilk 10 satiri goster

def net_anlasmazlik_isaretle(df):
    isimler = ["negatif", "nötr", "pozitif"]
    def ikinci_sinif_ve_margin(probs):
        sirali = sorted(range(3), key=lambda i: -probs[i])  # olasiliklari sirala
        return isimler[sirali[1]], probs[sirali[0]] - probs[sirali[1]]  # ikinci tercih ve farki dondur
    df["net_anlasmazlik"] = (
        df["sentiment_label"].isin(["negatif", "pozitif"]) &  # savasy kesin bir yon (negatif/pozitif) secmis
        df["sentiment_label_tbt_kalibre"].isin(["negatif", "pozitif"]) &  # TBT de kesin bir yon secmis
        (df["sentiment_label"] != df["sentiment_label_tbt_kalibre"])  # ama ikisi farkli yon secmis
    )
    return df  # anlasmazlik bayragi eklenmis dataframe'i dondur

df_clean = net_anlasmazlik_isaretle(df_clean)  # V2'de net anlasmazliklari bul
df_v5 = net_anlasmazlik_isaretle(df_v5)  # pilot v5'te net anlasmazliklari bul

print("df_clean net anlasmazlik:", df_clean["net_anlasmazlik"].sum())
print("df_v5 net anlasmazlik:", df_v5["net_anlasmazlik"].sum())

anlasmazlik_kolonlari = ["id", "text", "topic", "sentiment_label", "bert_score",
                          "sentiment_label_tbt_kalibre", "bert_score_tbt", "gercek_etiket"]  # excel'e yazilacak sutunlar

anlasmazliklar = pd.concat([
    df_clean[df_clean["net_anlasmazlik"]].assign(kaynak="V2"),  # V2'deki anlasmazlik satirlarini al, kaynak etiketi ekle
    df_v5[df_v5["net_anlasmazlik"]].assign(kaynak="pilot_v5"),  # pilot v5'teki anlasmazlik satirlarini al, kaynak etiketi ekle
])
anlasmazliklar["gercek_etiket"] = ""   # elle doldurman icin bos sutun

with pd.ExcelWriter("/Users/begumcetin/Desktop/sentiment analysis ito/ciktilar/anlasmazlik_tweetler.xlsx", engine="openpyxl") as writer:
    anlasmazliklar[["kaynak"] + anlasmazlik_kolonlari].to_excel(writer, index=False, sheet_name="Anlasmazliklar")  # excel'e yaz
    ws = writer.sheets["Anlasmazliklar"]
    ws.auto_filter.ref = ws.dimensions  # tum sutunlara filtre ekle
    ws.freeze_panes = "A2"  # baslik satirini sabitle

print("Kaydedildi: anlasmazlik_tweetler.xlsx")

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 57652.66it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: VRLLab/TurkishBERTweet
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


duzeltilen tweet sayisi: 12
duzeltilen tweet sayisi: 6
df_clean - savasy vs TurkishBERTweet (kalibre):
sentiment_label_tbt_kalibre  negatif  nötr  pozitif
sentiment_label                                    
negatif                           61   126       37
nötr                               4    32        7
pozitif                            3    58       39

df_v5 - savasy vs TurkishBERTweet (kalibre):
sentiment_label_tbt_kalibre  negatif  nötr  pozitif
sentiment_label                                    
negatif                           44    57       16
nötr                               2    16        5
pozitif                            3    38       23
df_clean - manuel kontrol gereken tweet sayisi: 115
df_v5 - manuel kontrol gereken tweet sayisi: 63

df_clean dikkat gerektirenlerden ornekler:
                                                                                                                                                                                           

## Bölüm 6 — Değerlendirme
*Detaylı accuracy/F1/confusion matrix değerlendirmesi, fine-tune edilmiş model için Bölüm 9.4'te yapılmıştır.*

## Bölüm 7 — Sonuç ve Çıktı Dosyası


In [45]:
# skor araligina gore filtrelenebilir bir bant sutunu olustur
def score_bandi(score):
  if score < 0.60:
      return "<%60"  # cok dusuk guven
  elif score < 0.70:
      return "%60-70"  # dusuk guven
  elif score < 0.80:
      return "%70-80"  # orta guven
  elif score < 0.90:
      return "%80-90"  # yuksek guven
  else:
      return "%90+"  # cok yuksek guven

df_clean["score_bandi"] = df_clean["bert_score"].apply(score_bandi)  # her tweet icin guven bandini hesapla
print(df_clean["score_bandi"].value_counts())  # bant dagilimini yazdir

score_bandi
%90+      222
%80-90     51
%60-70     41
%70-80     28
<%60       25
Name: count, dtype: int64


In [46]:
# id'ye gore eslesen ek dosyadan topic bilgisini geri getir
topic_df = pd.read_excel("/Users/begumcetin/Desktop/sentiment analysis ito/veri/X Tweet dataset.xlsx")[["id", "topic"]]  # id-topic eslesmesini iceren dosyayi oku

if "topic" in df_clean.columns:
    df_clean = df_clean.drop(columns=["topic"])  # varsa eski/bos topic sutununu sil

df_clean = df_clean.merge(topic_df, on="id", how="left")  # id uzerinden topic bilgisini df_clean'e ekle
print(df_clean["topic"].value_counts(dropna=False))  # topic dagilimini yazdir

topic
enflasyon    129
ulasim       123
turizm       115
Name: count, dtype: int64


In [47]:
cikti_sutunlari = [
    "id", "created_at", "text",          
    "processed_text",                     
    "bert_input_text",                    
    "sentiment_label",                   
    "bert_score",                        
    "score_bandi",           #yeni eklenen sütun
    "topic",
    "bert_raw_label",                    
    "sarcasm_suspicion",                   
    "hashtag_count", "retweet_count", "like_count", "reply_count", "quote_count" 
]

cikti_df = df_clean[cikti_sutunlari].copy()
cikti_df.head()

,id,created_at,text,processed_text,bert_input_text,sentiment_label,bert_score,score_bandi,topic,bert_raw_label,sarcasm_suspicion,hashtag_count,retweet_count,like_count,reply_count,quote_count
0,2054920537768443982,2026-05-14T13:43:29.000Z,"Lizzie,Londra ve İstanbul’daki fiyatları karş...",lizzielondra istanbul’daki fiyatları karşılaşt...,"Lizzie,Londra ve İstanbul’daki fiyatları karş...",nötr,0.606389,%60-70,enflasyon,negative,False,0,4,4,0,0
1,2054918753742114997,2026-05-14T13:36:24.000Z,İstanbul Bebek'te kapıcı dairesine talep edile...,bebekte kapıcı dairesine talep edilen yüksek k...,İstanbul Bebek'te kapıcı dairesine talep edile...,negatif,0.997620,%90+,enflasyon,negative,False,0,0,0,0,0
2,2054918432647160049,2026-05-14T13:35:07.000Z,Türk Hava Yolları İstanbul Havalimanı İç Hatla...,türk hava yolları havalimanı iç hatlar özel yo...,Türk Hava Yolları İstanbul Havalimanı İç Hatla...,nötr,0.647862,%60-70,turizm,positive,False,0,0,2,0,0
3,2054917617933029553,2026-05-14T13:31:53.000Z,İstanbul trafiği artık dayanılmaz bir hal aldı...,trafiği artık dayanılmaz hal aldı trafik,İstanbul trafiği artık dayanılmaz bir hal aldı...,negatif,0.959626,%90+,ulasim,negative,False,2,0,0,0,0
4,2054913996474425553,2026-05-14T13:17:29.000Z,"Değerli Taksi Esnafımız,\n\nTaksi sektörü, gel...",değerli taksi esnafımız taksi sektörü geleceği...,"Değerli Taksi Esnafımız,\n\nTaksi sektörü, gel...",pozitif,0.792551,%70-80,ulasim,positive,False,0,3,10,2,0


In [48]:
# excel'e yazarken filtreyi otomatik acik getir
with pd.ExcelWriter("/Users/begumcetin/Desktop/sentiment analysis ito/ciktilar/sentiment_analiz_sonuclari.xlsx", engine="openpyxl") as writer:
    cikti_df.to_excel(writer, index=False, sheet_name="Sonuclar")  # dataframe'i excel sayfasina yaz
    ws = writer.sheets["Sonuclar"]  # yazilan sayfaya referans al
    ws.auto_filter.ref = ws.dimensions   # tum sutunlara filtre oku ekler
    ws.freeze_panes = "A2"                # baslik satirini sabitler

print("Kaydedildi: sentiment_analiz_sonuclari.xlsx")

Kaydedildi: sentiment_analiz_sonuclari.xlsx


## Bölüm 8 — V2 ve Pilot v5 Verilerini Birleştirme


In [49]:
df_v5["score_bandi"] = df_v5["bert_score"].apply(score_bandi)   # pilot v5'e de guven bandi sutununu ekle

ortak_sutunlar = [
    "id", "created_at", "text", "processed_text", "bert_input_text",
    "sentiment_label", "bert_score", "score_bandi", "topic", "bert_raw_label",
    "sarcasm_suspicion", "hashtag_count", "retweet_count", "like_count", "reply_count", "quote_count"
]  # iki veri setinde de ortak olan, birlestirmede kullanilacak sutunlar

df_clean_etiketli = cikti_df.copy()  # V2'nin son halini kopyala
df_clean_etiketli["kaynak"] = "V2"  # hangi veri setinden geldigini isaretle

df_v5_etiketli = df_v5[ortak_sutunlar].copy()  # pilot v5'ten sadece ortak sutunlari al
df_v5_etiketli["kaynak"] = "pilot_v5"  # hangi veri setinden geldigini isaretle

df_birlesik = pd.concat([df_clean_etiketli, df_v5_etiketli], ignore_index=True)  # iki veri setini alt alta birlestir
print("V2:", len(df_clean_etiketli), "+ pilot_v5:", len(df_v5_etiketli), "= toplam:", len(df_birlesik))
print(df_birlesik["kaynak"].value_counts())  # kaynaklara gore satir sayisini yazdir
print(df_birlesik["sentiment_label"].value_counts())  # sinif dagilimini yazdir

with pd.ExcelWriter("/Users/begumcetin/Desktop/sentiment analysis ito/ciktilar/df_birlesik.xlsx", engine="openpyxl") as writer:
    df_birlesik.to_excel(writer, index=False, sheet_name="Sonuclar")  # birlesik veriyi excel'e yaz
    ws = writer.sheets["Sonuclar"]
    ws.auto_filter.ref = ws.dimensions  # tum sutunlara filtre ekle
    ws.freeze_panes = "A2"  # baslik satirini sabitle

print("Kaydedildi: df_birlesik.xlsx")

V2: 367 + pilot_v5: 204 = toplam: 571
kaynak
V2          367
pilot_v5    204
Name: count, dtype: int64
sentiment_label
negatif    341
pozitif    164
nötr        66
Name: count, dtype: int64
Kaydedildi: df_birlesik.xlsx


## Bölüm 9 — savasy Modelinin Gold-Standard Veriyle Fine-Tune Edilmesi


## Bölüm 9.1 — Fine-Tuning için Veri Hazırlığı

In [50]:
import torch
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW

gold_df = pd.read_excel("/Users/begumcetin/Desktop/sentiment analysis ito/ciktilar/gold_standard_208.xlsx")  # elle etiketlenmis 208 satirlik gold veriyi oku
label2id = {"negatif": 0, "nötr": 1, "pozitif": 2}  # metin etiketini sayisal id'ye cevirmek icin sozluk
id2label_ft = {v: k for k, v in label2id.items()}  # tersini de tutar (id'den etikete)
gold_df["label"] = gold_df["gercek_etiket"].map(label2id)  # gercek_etiket sutununu sayisal label'a cevir

train_df, val_df = train_test_split(
    gold_df, test_size=0.2, random_state=42, stratify=gold_df["label"]
)  # veriyi sinif dengesini koruyarak %80 train / %20 validation olarak bol
print("train:", len(train_df), "val:", len(val_df))  # her iki parcanin satir sayisini yazdir

train: 166 val: 42


## Bölüm 9.2 — Model ve LoRA Kurulumu

In [51]:
model_adi = "savasy/bert-base-turkish-sentiment-cased"  # taban alinacak modelin adi
ft_tokenizer = AutoTokenizer.from_pretrained(model_adi)  # modelin tokenizer'ini yukle
# num_labels=3 verince orijinal 2 sinifli classifier basi atilip
# yerine 3 sinifli (negatif/notr/pozitif) yeni bir bas takiliyor
ft_model = AutoModelForSequenceClassification.from_pretrained(
    model_adi, num_labels=3, ignore_mismatched_sizes=True
)  # modeli 3 sinifli yeni bir classifier basiyla yukle

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, r=8, lora_alpha=16, lora_dropout=0.1,
    target_modules=["query", "value"],
)  # LoRA ayarlari: sadece query/value katmanlarina kucuk, egitilebilir adaptorler eklenecek
ft_model = get_peft_model(ft_model, lora_config)  # LoRA adaptorlerini modele bindir
ft_model.print_trainable_parameters()  # toplamda kac parametrenin egitilecegini yazdir (cok az olmasi beklenir)

[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6797.90it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: savasy/bert-base-turkish-sentiment-cased
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([2, 768]) vs model:torch.Size([3, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([2]) vs model:torch.Size([3])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


trainable params: 297,219 || all params: 110,916,870 || trainable%: 0.2680


## Bölüm 9.3 — Eğitim Döngüsü

In [52]:
class TweetDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = list(texts); self.labels = list(labels)  # metin ve etiket listelerini sinif icinde sakla
    def __len__(self): return len(self.texts)  # veri setindeki toplam ornek sayisini dondur
    def __getitem__(self, i): return self.texts[i], self.labels[i]  # i. indeksteki (metin, etiket) ciftini dondur

def collate(batch):
    texts, labels = zip(*batch)  # batch icindeki (metin,etiket) ciftlerini ayri listelere ayir
    enc = ft_tokenizer(list(texts), return_tensors="pt", truncation=True, max_length=128, padding=True)  # metinleri tokenize edip tensor'a cevir, kisa olanlari padle
    enc["labels"] = torch.tensor(labels)  # etiketleri de tensor olarak ekle
    return enc  # modele direkt verilebilecek hazir batch'i dondur

train_loader = DataLoader(TweetDataset(train_df["text"], train_df["label"]), batch_size=8, shuffle=True, collate_fn=collate)  # egitim verisini 8'lik gruplar halinde, her epoch'ta karistirarak yukleyen loader
val_loader = DataLoader(TweetDataset(val_df["text"], val_df["label"]), batch_size=8, shuffle=False, collate_fn=collate)  # validation verisini karistirmadan yukleyen loader

# sinif dengesizligine karsi (negatif fazla) agirlikli loss
sinif_sayilari = train_df["label"].value_counts().sort_index()  # egitim setindeki her sinifin kac ornegi oldugunu say
agirliklar = torch.tensor([len(train_df) / (3 * sinif_sayilari[i]) for i in range(3)], dtype=torch.float)  # az ornekli siniflara daha yuksek agirlik ver
loss_fn = torch.nn.CrossEntropyLoss(weight=agirliklar)  # sinif agirlikli kayip fonksiyonunu tanimla

optimizer = AdamW(ft_model.parameters(), lr=2e-4)  # LoRA parametrelerini guncelleyecek optimizer

for epoch in range(8):  # 8 tur (epoch) boyunca egit
    ft_model.train()  # modeli egitim moduna al
    toplam_loss = 0  # bu epoch'un toplam kaybini tutacak degisken
    for batch in train_loader:  # egitim verisini batch batch gez
        optimizer.zero_grad()  # onceki adimin gradyanlarini sifirla
        labels = batch.pop("labels")  # gercek etiketleri batch'ten ayir
        out = ft_model(**batch)  # modeli calistirip logit'leri al
        loss = loss_fn(out.logits, labels)  # agirlikli kaybi hesapla
        loss.backward()  # gradyanlari geriye yayilim ile hesapla
        optimizer.step()  # model parametrelerini guncelle
        toplam_loss += loss.item()  # bu batch'in kaybini topla

    ft_model.eval()  # modeli degerlendirme moduna al (dropout vs kapanir)
    tum_pred, tum_gercek = [], []  # tahminleri ve gercek etiketleri toplayacak listeler
    with torch.no_grad():  # degerlendirmede gradyan hesaplama, hizli olsun
        for batch in val_loader:  # validation verisini batch batch gez
            labels = batch.pop("labels")  # gercek etiketleri ayir
            out = ft_model(**batch)  # modeli calistir
            tum_pred.extend(out.logits.argmax(-1).tolist())  # en yuksek olasilikli sinifi tahmin olarak al
            tum_gercek.extend(labels.tolist())  # gercek etiketleri de listeye ekle
    acc = np.mean(np.array(tum_pred) == np.array(tum_gercek))  # dogru tahmin oranini hesapla
    print(f"epoch {epoch+1} | train_loss={toplam_loss/len(train_loader):.3f} | val_acc={acc:.3f}")  # bu epoch'un sonucunu yazdir

epoch 1 | train_loss=0.993 | val_acc=0.548
epoch 2 | train_loss=0.847 | val_acc=0.619
epoch 3 | train_loss=0.719 | val_acc=0.762
epoch 4 | train_loss=0.574 | val_acc=0.762
epoch 5 | train_loss=0.456 | val_acc=0.810
epoch 6 | train_loss=0.410 | val_acc=0.714
epoch 7 | train_loss=0.327 | val_acc=0.810
epoch 8 | train_loss=0.271 | val_acc=0.810


## Bölüm 9.4 — Değerlendirme ve Kaydetme

In [53]:
print(classification_report(tum_gercek, tum_pred, target_names=["negatif", "nötr", "pozitif"]))  # son epoch'un detayli basari raporunu yazdir (precision/recall/f1)
ft_model.save_pretrained("/Users/begumcetin/Desktop/sentiment analysis ito/model/savasy_lora_finetuned")  # egitilmis LoRA adaptorunu diske kaydet
print("Kaydedildi: savasy_lora_finetuned/")

              precision    recall  f1-score   support

     negatif       0.86      0.90      0.88        20
        nötr       0.89      0.62      0.73        13
     pozitif       0.67      0.89      0.76         9

    accuracy                           0.81        42
   macro avg       0.80      0.80      0.79        42
weighted avg       0.83      0.81      0.81        42

Kaydedildi: savasy_lora_finetuned/


## Bölüm 9.5 — Fine-Tune Edilmiş Modeli Tüm Veriye Uygulama

In [ ]:
from peft import PeftModel  # egitilmis LoRA adaptorunu yuklemek icin

base_model_infer = AutoModelForSequenceClassification.from_pretrained(
    model_adi, num_labels=3, ignore_mismatched_sizes=True
)  # savasy'nin taban modelini 3 sinifli bos bir classifier basiyla yeniden yukle
ft_model_infer = PeftModel.from_pretrained(
    base_model_infer, "/Users/begumcetin/Desktop/sentiment analysis ito/model/savasy_lora_finetuned"
)  # kaydedilmis LoRA adaptorunu taban modelin uzerine bindir
ft_model_infer.eval()  # modeli degerlendirme moduna al

def ft_tahmin_et(text):
    enc = ft_tokenizer(str(text), return_tensors="pt", truncation=True, max_length=128)  # metni tokenize et
    with torch.no_grad():
        logits = ft_model_infer(**enc).logits  # fine-tune edilmis modeli calistir
    return id2label_ft[logits.argmax(-1).item()]  # en yuksek olasilikli sinifin etiketini dondur

df_birlesik["sentiment_label_ft"] = df_birlesik["bert_input_text"].apply(ft_tahmin_et)  # tum birlesik veriye fine-tune edilmis modeli uygula

print("eski (savasy) vs yeni (fine-tuned):")
print(pd.crosstab(df_birlesik["sentiment_label"], df_birlesik["sentiment_label_ft"]))  # eski ve yeni etiketleri karsilastiran capraz tablo
print()
print(df_birlesik["sentiment_label_ft"].value_counts())  # yeni etiketin sinif dagilimini yazdir

with pd.ExcelWriter("/Users/begumcetin/Desktop/sentiment analysis ito/ciktilar/sentiment_analiz_sonuclari_final.xlsx", engine="openpyxl") as writer:
    df_birlesik.to_excel(writer, index=False, sheet_name="Sonuclar")  # dataframe'i excel'e yaz
    ws = writer.sheets["Sonuclar"]  # yazilan sayfaya referans al
    ws.auto_filter.ref = ws.dimensions  # tum sutunlara otomatik filtre ekle
    ws.freeze_panes = "A2"  # baslik satirini sabitle

print("Kaydedildi: sentiment_analiz_sonuclari_final.xlsx")

[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 12878.74it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: savasy/bert-base-turkish-sentiment-cased
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([2, 768]) vs model:torch.Size([3, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([2]) vs model:torch.Size([3])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


eski (savasy) vs yeni (fine-tuned):
sentiment_label_ft  negatif  nötr  pozitif
sentiment_label                           
negatif                 256    69       16
nötr                     14    41       11
pozitif                  18    59       87

sentiment_label_ft
negatif    288
nötr       169
pozitif    114
Name: count, dtype: int64
Kaydedildi: sentiment_analiz_sonuclari_final.xlsx


## Sonuç Özeti

**Veri:** V2 (367 tweet) + Pilot v5 (204 tweet) = 571 tweet, turizm/ulaşım/enflasyon konularında, İstanbul'a yönelik.

**Model karşılaştırması:** savasy, saribasmetehan ve TurkishBERTweet modelleri 208 satırlık elle etiketlenmiş gold-standard veri üzerinde karşılaştırıldı. savasy, özellikle negatif içerik tespitinde (%86.6) diğerlerinden üstün bulundu ve taban model olarak korundu.

**Fine-tuning:** savasy modelinin tek zayıf noktası olan nötr sınıf tespiti (haber/duyuru formatlı tweetlerin yanlış etiketlenmesi), 208 satırlık gold-standard veriyle LoRA yöntemiyle fine-tune edilerek düzeltildi.

| Metrik | Fine-tuning Öncesi | Fine-tuning Sonrası |
|---|---|---|
| Genel doğruluk | %61.5 | %81 |
| Negatif yakalama | %86.6 | %90 |
| Nötr yakalama | %23.9 | %62 |
| Pozitif yakalama | %63.6 | %89 |

**Nihai çıktı:** `ciktilar/sentiment_analiz_sonuclari_final.xlsx` — 571 tweetin hepsi için fine-tune edilmiş modelin ürettiği `sentiment_label_ft` sütununu içerir.